In [22]:
from IPython.display import display, HTML
import datetime

display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [6]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
db = duckdb.connect()

from ticks_collector.kite_utils import get_initialized_kite

In [9]:
setup_query = """
INSTALL httpfs;

LOAD httpfs;

INSTALL aws;
LOAD aws;

CREATE OR REPLACE SECRET secret (TYPE s3, PROVIDER credential_chain, REGION 'ap-south-1');
"""

db.sql(setup_query).execute()


┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [10]:
parquet_dir = "s3://ticks-data-bucket/ticks/**/*.parquet"

In [11]:
db.sql(f"""
CREATE OR REPLACE VIEW nse_ticks AS
SELECT * FROM parquet_scan('{parquet_dir}');
""")

In [7]:
kite = get_initialized_kite()
ins = pd.DataFrame(kite.instruments(exchange="NSE"))

In [12]:
def get_instrument_token_id(symbol):
    return int(ins.loc[ins.symbol == symbol, "instrument_token"].iat[0])


In [13]:
db.sql("PRAGMA table_info('nse_ticks');")

┌───────┬──────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────┬────────────┬─────────┐
│  cid  │         name         │                                                              type                                                               │ notnull │ dflt_value │   pk    │
│ int32 │       varchar        │                                                             varchar                                                             │ boolean │  varchar   │ boolean │
├───────┼──────────────────────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┼─────────┼────────────┼─────────┤
│     0 │ tradable             │ BOOLEAN                                                                                                                         │ false   │ NULL       │ false   │
│     1 │ mode      

'2025-12-29'

In [20]:
today_ticks = db.sql(f"""
SELECT * from nse_ticks

WHERE date = '{datetime.date.today().strftime("%Y-%m-%d")}'
""")

In [30]:
today_ins = today_ticks.project("instrument_token").distinct()

In [31]:
df = today_ins.to_df()

In [33]:
df

,instrument_token
0,7343873
1,272393
2,267273
3,417289
4,6928897
...,...
8767,7485697
8768,2624513
8769,5317633
8770,5938177


In [41]:
ts = ins[ins.lot_size > 1].tradingsymbol.tolist()

In [46]:
map(lambda x: len(x.split("-")) == 2, ts))

True